## Set up

In [1]:
import csv, json, requests, re, ast, copy
from pprint import pprint
import numpy as np
from scipy import spatial
from tqdm import tqdm
from openpyxl import load_workbook
import pandas as pd
import html
import pandas as pd
import re, html, json, ast

In [2]:
from typing import List, Dict, Any, Optional, Tuple
from pathlib import Path
from langchain_core.output_parsers import JsonOutputParser

In [3]:
#used for decision tree generation
import xml.etree.ElementTree as ET
from xml.dom import minidom
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [4]:
from langchain_openai import AzureChatOpenAI
api_key="4d78888ae7d34d38bf5a0ec97a69a6f6"
llm = AzureChatOpenAI(
    api_key=api_key,
    azure_endpoint="https://genaitraining-aoai2.openai.azure.com/",
    azure_deployment="gpt-4o",  # This must match your Azure deployment name
    api_version="2024-12-01-preview",  # Or "2025-01-01-preview" if that's correct
    temperature=0
)

#from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

#llm = ChatOpenAI(model="gpt-4o", temperature=0, api_key=api_key)


############ Used for chaining
from langchain.prompts import PromptTemplate
#from langchain_core.runnables.base import RunnableSequence




def get_completion(prompt, model=llm):
    response = model.invoke([HumanMessage(content=prompt)])
    return response.content


############## Embedding Creation
from openai import AzureOpenAI

client = AzureOpenAI(
  api_key = api_key,  
  api_version = "2024-10-21",
  azure_endpoint ="https://genaitraining-aoai2.openai.azure.com/"
)

def get_embedding_vector(text):
    embedding_response = client.embeddings.create(
        input = text,
        #model= "text-embedding-3-large"
        model= "text-embedding-ada-002"
    )

    embedding_response_dict = embedding_response.model_dump_json()
    parsed_embedding_respomde_dict = json.loads(embedding_response_dict)
    embedding_vector = parsed_embedding_respomde_dict['data'][0]['embedding']

    return embedding_vector


In [5]:
def write_lists_csv(list1,list2,header1,header2, output_filename):
   
    with open(output_filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([header1, header2])  # Write header row
        max_length = max(len(list1), len(list2))
        for i in range(max_length):
            row = [list1[i] if i < len(list1) else '', list2[i] if i < len(list2) else '']
            writer.writerow(row)  # Write data rows
    return

In [6]:
import re
import ast
import html
import json

def extract_json_string(text: str):
    # 1) Trim and strip code fences (``` or ```json)
    t = text.strip()
    t = re.sub(r'^\s*```(?:json)?\s*|\s*```\s*$', '', t, flags=re.IGNORECASE)

    # 2) If the entire payload is a quoted Python string literal, unquote it safely
    if (t.startswith("'") and t.endswith("'")) or (t.startswith('"') and t.endswith('"')):
        try:
            t = ast.literal_eval(t)
        except Exception:
            pass  # Continue with t as-is if unquoting fails

    # 3) Decode HTML entities
    t = html.unescape(t)

    # 4) Normalize Python-ish literals to JSON
    t = t.replace("True", "true").replace("False", "false").replace("None", "null")

    # 5) Remove excessive escaping (e.g., \\n, \\t, \\\")
    try:
        t = t.encode().decode('unicode_escape')
    except Exception as e:
        print(f"Unicode escape decoding failed: {e}")

    # 6) Pre-validate JSON structure
    if not (t.strip().startswith('{') or t.strip().startswith('[')):
        print("Warning: Input does not start with a valid JSON object or array.")

    # 7) Try parsing as JSON
    try:
        return json.loads(t)
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error: {e}")
        print("Attempting fallback parsing...")

        # 8) Fallback: extract the first top-level array/object and parse that
        m = re.search(r'(\{.*?\}|\[.*?\])', t, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError as e2:
                print(f"Fallback JSON Decode Error: {e2}")

        # Optional: use permissive parser if available
        # import demjson3
        # return demjson3.decode(t)

        raise e  # Re-raise original error if all else fails

## Medical Document Ingestion

In [7]:
def read_lcd_from_excelfile(lcd_id, excelfile_path = './LCDs/lcd.xlsx'):
    # Load the workbook
    workbook = load_workbook(filename=excelfile_path)

    # Select a specific sheet
    sheet = workbook['lcd']  # Change to your desired sheet name


    # Read header row to find column indices
    header = [cell.value for cell in sheet[1]]
    try:
        
        lcd_id_index = header.index('lcd_id')
        indication_index = header.index('indication')
        title_index = header.index('title')

    except ValueError:
        raise Exception("Required columns not found in the header.")

    for row in sheet.iter_rows(min_row=2, values_only=True):  # Skip header
        if row[lcd_id_index] == lcd_id:
            content = row[indication_index]
            procedure_title = row[title_index]
            return content, procedure_title


    else:
        print(f"lcd_id {lcd_id} not found.")

        return None


In [29]:
def fetch_MCG(filename):

    with open(filename, "r", encoding="utf-8") as file:
        html_content = file.read()

    
    # Match the start of the 'Clinical Indications' section
    start_match = re.search(r'<h[23].*?>\s*Clinical Indications.*?</h[23]>', html_content, re.IGNORECASE)
    
    # used for debugging
    #titles = re.findall(r'<h[23].*?>\s*(.*?)\s*</h[23]>', html_content, re.IGNORECASE)
    #print(f"titles = {titles}")


    
    # Match the next title after 'Clinical Indications'
    next_title_match = None
    if start_match:
        start_pos = start_match.end()
        next_title_match = re.search(r'<h[23].*?>.*?</h[23]>', html_content[start_pos:], re.IGNORECASE)
        end_pos = start_pos + next_title_match.start() if next_title_match else len(html_content)

        # Extract content between 'Clinical Indications' and the next title
        MCG_content = html_content[start_pos:end_pos]
    else:
        MCG_content = "Clinical Indications section not found."



    # Extract the title of the procedure
    title_pattern = r"<title[^>]*>(.*?)</title>"
    match_title = re.search(title_pattern, html_content, re.DOTALL)

    # Extract the LCD ID
    MCG_ID_pattern = r'<p[^>]*class="HSIM"[^>]*>(.*?)</p>'
    MCG_ID_match = re.search(MCG_ID_pattern, html_content)

    # Extract the relevant content
    #MCG_content = match.group(1) if match else None
    MCG_title = html.unescape(match_title.group(1) )if match_title else None
    MCG_ID = MCG_ID_match.group(1) if MCG_ID_match else None

    return MCG_content, MCG_title, MCG_ID

In [30]:
# Testing MCG's

#filename = 'AC - Aflibercept.html'
filename = 'AC - Blepharoplasty, Canthoplasty, and Related Procedures.html'
#filename = 'AC - Myocardial Positron Emission Tomography (PET) and PET-CT.html'
#filename = 'AC - Denosumab A-044.html'

In [31]:
def fetch_LCD(url):
    response = requests.get(url)
    
    # to avoids encoding errors and replaces any problematic characters with safe placeholders
    text = response.content.decode('utf-8', errors='replace')


    # In order to decrease the size of the text, we will extract only the content between "Coverage Guidance" and "General Information"
    # This will help us focus on the relevant part of the document for further processing and lessen the token count for the LLM.
    pattern = r'Coverage Guidance</h3>(.*?)General Information</h2>'
    match = re.search(pattern, text, re.DOTALL)
    content = match.group(1) if match else None

    # Extract the title of the procedure
    title_pattern = r'<title>LCD\s+-\s+(.*?)\s*\(.*?\)</title>'
    match_title = re.search(title_pattern, text, re.DOTALL)
    LCD_title = html.unescape(match_title.group(1) )if match_title else None

    # Extract the LCD ID
    LCD_ID_pattern = r'<title>LCD\s+-\s+.*?\((.*?)\)</title>'
    LCD_ID_match = re.search(LCD_ID_pattern, text)
    LCD_ID = LCD_ID_match.group(1) if LCD_ID_match else None

    
    

    return content, LCD_title, LCD_ID

## Medical Guidelines Extraction

In [32]:
# This cell is used for testing.
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?LCDId=34635' #Botulinum Toxin Type A & Type B
url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?LCDId=36573&ContrId=345' #THA
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?LCDId=36575' #TKA
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35172&ver=68&bc=0'
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35172&ver=68&bc=0'
#url = 'https://www.cms.gov/medicare-coverage-database/view/lcd.aspx?lcdid=35172&ver=68&bc=0'


LCD, LCD_title, LCD_ID = fetch_LCD(url)
name = LCD_title


In [33]:
print(LCD_title)
print(LCD_ID)

Total Hip Arthroplasty
L36573


In [35]:
# MCGs = ['AC - Myocardial Perfusion Imaging, Pharmacologic Stress A-0079.html',
#         'AC - Abdominal_Pelvic CT Angiography (CTA) A-0475.html',
#         'AC - Aflibercept.html',
#         'AC - Cervical Spine MRI A-0057.html',
#         'ISC - Lumbar Fusion S-820.html',
#         'AC - Cervical Spine MRI A-0057.html',
#         'AC - Aflibercept.html',
#         'AC - Blepharoplasty, Canthoplasty, and Related Procedures.html',
#         'AC - Myocardial Positron Emission Tomography (PET) and PET-CT.html']

# MCG, MCG_title, MCG_ID = fetch_MCG(MCGs[0])

In [36]:
def extract_LCD_medicalguidelines(LCD, LCD_title):

  instruct_template = """

    Instruct: You are expert in prior authorization in health insurance companies. Your job is extract all medical guidelines, including indications\

    and limitations, from a Local Coverage Determination (LCD) document. Here is their definition:

      - Indications: These are the clinical scenarios, diagnoses, or patient conditions under which a service, procedure, or item is considered \
          reasonable and necessary and therefore eligible for Medicare coverage.

      - Limitations: These are the boundaries or restrictions placed on the use of a service, even when it is generally covered. They define how \
          often, under what conditions, or for which patient populations the service is not covered.

      Please pay attention we don't need any other information in LCD's beyond indications and limitations. Please keep the definition of indications \
      and limitations for next prompts.
      In continue, I am going to ask you to extract some information from an LCD document which is in html format. Please don't respond to this prompt and wait for next prompts.
  """

  contraindications_extraction_prompt = """
  thought:
    In medicine, a contraindication is a condition (a situation or factor) that serves as a reason not to take a certain medical treatment 
    or procedure due to the harm that it would cause the patient. Contraindication is the opposite of indication, which is a reason to use a 
    certain procedure or treatment, so contraindication is a reason to not use a certain procedure or treatment.

    Contraindications are distinct from limitations. Limitations refer to coverage boundaries such as frequency, dosage, or quantity restrictions, 
    and should not be extracted. Only extract contraindications that relate to clinical safety concerns. Don't consider limitations.

    Contraindications can be classified into two main categories based on how they are presented in the document:
    - "explicit" contraindications: typically presented in structured formats such as:
      - Bullet points (<ul><li>) or (&bull;)
      - Numbered or alphabetically ordered lists (<ol type="1"|"A"|"a">)

    - "implicit" contraindications: refer to those that are not explicitly listed in structured formats such as bullet points, numbered lists, 
      or tables. Instead, they are embedded within narrative paragraphs of the document, often requiring careful interpretation to 
      extract relevant clinical meaning.

    Alongside exrtacting contraindications, it's equally important to identify the **clinical context**.
    Clinical context typically:
        - Provides background, rationale, or framing for contraindications that follow.
        - Appears as plain text or paragraph(s), or as title section before explicit guidelines.
        - May include definitions, epidemiological data, treatment overview, or references to clinical guidelines.
        - A clinical context belongs to all contraindication coming after, so the clinical context should be assigned to all related contraindications.
      

  action:
  Extract all contraindications of {LCD_title} from {LCD}.
  Return the extracted contraindications as a **valid JSON array of objects**. Don't return any limitation. Each object should follow this format:

    [{{'contraindication ID': create an ID like this Cont_1,
        'contraindication': put the contraindication here,
        'explicit': put "True" if the contraindication is explicit; otherwise put "False"
        'reason': provide a short explanation for implicit contraindications why you picked that statement as a contraindication. For
        explicit contraindications just put 'None'.
        'clinical_context': put `clinical context` here. If there is no clinical context, just put `None`.
    }}]
    

  Use **double quotes** for all keys and string values. Use `True`, `False`, and `Null` for boolean and null values.
  Do not wrap the output in triple backticks or return it as a string. Just return the raw JSON array.

  Each object should include:
  - "contraindication_ID"
  - "contraindication"
  - "explicit"
  - "reason"
  - "clinical_context"

  Use valid JSON syntax with double quotes and JSON-native values (True, False, Null). Do not return a raw list or wrap the output in markdown.

        """

  explicit_mdgs_extraction_prompt = """ 
  thought:
  "Explicit medical guidelines" are typically presented in structured formats such as:
  - Bullet points (<ul><li>) or (&bull;)
  - Numbered or alphabetically ordered lists (<ol type="1"|"A"|"a">)
  - Nested lists of any depth, including combinations of <ul> and <ol> tags.

  These structures may contain multiple levels of nesting, where:
  - A top-level <li> may contain a nested <ol> or <ul>,
  - Each nested list may contain further <li> elements, and so on.

  The HTML may include a mix of list types (e.g., numeric, alphabetic, symbolic), and the nesting may go beyond three levels.

  action:
  Extract all **explicit** medical indications and limitations from {LCD}, a Local Coverage Determination (LCD) document in HTML format.

  Your task must:
  - Traverse all <li> elements, including those nested within <ul> or <ol> tags of any depth.
  - Recursively extract content from nested lists, preserving the full hierarchical structure.
  - Include all structured content, even if it appears under <ol type="A">, <ol type="a">, <ul style="list-style-type: circle;">, or similar.
  - Represent each guideline as a dictionary. If a guideline contains sub-guidelines (e.g., diagnostic criteria, symptoms, or conditions), include them as a nested list under a key called "sub_mdgs". Each sub-guideline should follow the same dictionary format:

              mdg_ID: Use hierarchical IDs like mdg_7.1, mdg_7.2, etc.
              mdg: Text of the sub-guideline.
              type: Same as parent unless specified.
              explicit: put True
              justification: put None
              extraction time: Same as parent.
  - Ignore narrative text outside of structured list elements unless it introduces a list that follows.
      Do not extract introductory or transitional phrases that precede a list (e.g., “will be considered medically necessary in the following circumstances” or “demonstrated by:”).
  - Only extract content that is within <li> elements or clearly structured as a guideline, not standalone headers or lead-ins.
  - If a sentence introduces a list, ignore it unless it is part of a <li> element.

  response format:
  Return the list of medical guidelines, including indications and limitations, without any extra explanation. Each guideline should be organized in a dictionary format with the following keys:
  - mdg_ID: Use sequential IDs like mdg_1, mdg_2, etc.
  - mdg: The extracted medical guideline exactly as it appears in the LCD document.
  - type: Either "indication" or "limitation".
  - explicit: True
  - justification: None
  - extraction time: Current date and time in the format YYYY-MM-DD HH:MM:SS.


  Use **double quotes** for all keys and string values. Use `True`, `False`, and `Null` for boolean and null values.
  Do not wrap the output in triple backticks or return it as a string. Just return the raw JSON array.

  observation:
  Here are examples of explicit indications and limitations from an LCD in HTML format:
  - <li>anti-inflammatory medications or analgesics, or</li>
  - • Loosening of one or both components; or<br /><br />•
  - Multi-level example:
    <li>Chronic migraine is defined as... Treatment of chronic migraines will be covered when they meet the following diagnostic criteria:
      <ol type="A">
        <li><strong>Migraine with aura:</strong>
          <ol>
            <li>At least two attacks fulfilling the following criteria:
              <ol type="a">
                <li><strong>One or more of the following reversible aura symptoms:</strong>
                  <ul>
                    <li>Visual (aura, changes in vision)</li>
                    <li>Sensory (e.g., tingling in hands or face)</li>
                    ...
                  </ul>
                </li>
              </ol>
            </li>
          </ol>
        </li>
      </ol>
    </li>


Do not include any introductory text, explanation, or markdown formatting. Just return the raw JSON array.
          """          
              
  implicit_mdgs_extraction_prompt = """ 

  reminder:  Indications are the clinical scenarios, diagnoses, or patient conditions under which a service, procedure, or item is considered reasonable and necessary and therefore eligible for Medicare coverage.

  thought:
          Implicit indications, refer to those that are not explicitly listed in structured formats such as bullet points, numbered lists, or tables. Instead, they are embedded within narrative paragraphs of the document, often requiring careful interpretation to extract relevant clinical meaning.

          Indications are considered implicit only if there is no semantically identical or equivalent explicit indication listed in the structured\
        sections that follow the paragraph. If a similar indication is later presented explicitly, the narrative mention is treated as contextual\
            reinforcement rather than a distinct implicit rule.,so you shuldn't consider it as an implicit indication.


      For implicit guidelines, the sentence or paragraph, that the implicit guideline is taken from, is considered as 'clinical guideline'
      
  action:
  Your task is to extract implicit medical indications and limitations from {LCD}, which is a Local Coverage Determination (LCD) document in html format. 
  you should respond to this prompt with a list of medical guidelines without any extra explanation.\
  Each indication and limitation should be organized in a dictionary format with the following keys:
      - mdg_ID: put an ID like mdg_1 and mdg_2, etc.,
      - mdg: put here the extracted indication exactly as it is in the LCD document,
      - type: put the type of the medical guideline here as it is indication or limitation.
      - explicit: put False if the medical guideline is implicit, otherwise put True.
      - justification: Please provide a brief justification why you considered this indication as an implicit indication.
      - extraction time: put the current date and time in the format YYYY-MM-DD HH:MM:SS.

  Use **double quotes** for all keys and string values. Use true`, `false`, and `null` for boolean and null values.
  Do not wrap the output in triple backticks or return it as a string. Just return the raw JSON array.


  observation:
  Here is a paragraph with a set of explicit indications afterward:
  <p>In some circumstances, for example, if the patient has bone on bone articulation, severe deformity, pain or significant disabling interference with activities of daily living, the surgeon may determine that nonsurgical medical management would be ineffective or counterproductive and that the best treatment option, after explaining the risks, is surgical. If medical management is deemed appropriate, the medical record should indicate the rationale for and the circumstances under which this is the case.<br /><br /></p>
  <ul>
  <li>Malignancy of the joint involving the bones or soft tissues of the pelvis or proximal femur; <strong> or</strong></li>
  </ul>
  <ul>
  <li>Avascular necrosis (osteonecrosis of femoral head); <strong> or</strong></li>
  </ul>
  <ul>
  <li>Fracture of the femoral neck; <strong> or</strong></li>
  </ul>
  <ul>
  <li>Acetabular fracture; <strong> or</strong></li>
  </ul>
  <ul>
  <li>Non-union or failure of previous hip fracture surgery; <strong> or</strong></li>
  <li>Mal-union of acetabular or proximal femur fracture</li>
  </ul>

  "bone on bone articulation" is an implicit indications that is presented in the paragraph, but it is not listed in the explicit indications.

Do not include any introductory text, explanation, or markdown formatting. Just return the raw JSON array.
          """          

  consolidating_mdgs_prompt = """
  thought:
  You are consolidating medical guidelines extracted from an LCD document.

  Given the following two sets of extracted guidelines:

  Explicit Guidelines:
  {explicit_mdgs_extracted}

  Implicit Guidelines:
  {implicit_mdgs_extracted}

  action:
  Your task is to merge them into a unified list of medical guidelines in the keys:

  - mdg_ID
  - mdg
  - type
  - explicit
  - justification
  - extraction_time

  Please put None if there is no justification for the justification key.

  When merging implicit guidelines into the list of explicit guidelines, assign each implicit guideline a new mdg_ID that continues the numbering \
      from the last explicit guideline.
  Please use the values if each medical guideline as they are in the Explicit Guidelines and in the implicit guidlines.
  Use **double quotes** for all keys and string values. Use `True`, `False`, `None` and `Null` for boolean and none or null values.
  Do not wrap the output in triple backticks or return it as a string. Just return the raw JSON array.

  """

  structure_extraction_prompt = """
  thought:
  In order to capture the complexity of the hierarchical structure of medical guidelines, you have to consider the definition of specific terms as below:
    - parent groups: Indications and limitations are often organized into distinct groups separated by a paragraph, sentence, or section title.
    - parent_id: it is a unique identifier generated for each parent group. pattern_id is like pg_1, pg_2, and so on.
    - child group:  Within a parent group, there may be standalone medical guidelines as well as nested sets of related items, which we will call 'child groups'. Each child group contains sub-indications or sub-limitations that expand on the parent group’s content.
    - child_group_id: it is a unique identifier generated for each child group. child_group_id is like chg_1, chg_2, and so on.
    - type: refers to thr type of medical guideline which is either indication or limitation. 
    - logical relation: refers to those logical AND's or OR's often exist between items within a parent group and its child groups. These logical connectors are typically stated at the end of indications and limitations. If no logical relation is explicitly stated at the end of an explicit indication or explicit limitation in the LCD document—or if you have considered an implicit indication or implicit limitation, please follow these rules:
              Rule 1: If an item is an indication, assume its logical relation is OR.
              Rule 2: If an item is a limitation, assume its logical relation is AND.
    - mdgs: refer to the list of medical guidelines of a specific parent group. Each medical guideline will be represented as a dictionary whose keys are member_id, and content.
    - member_id: is a unique identifer which is created based on parent_id. For example, for pg_1, the identifer of parent group 1, member_id's are mdg_1.1, mdg_2.2, and so on. Likewise, for pg_2, the identifier of parent group 2, member_id's are mdg_2.1, mdg_2.2, and so on.
    - content: is the body of a medical guideline extracted from the LCD.
    - Alongside extracting explicit guidelines, it is equally important to identify the clinical context.

    - clinical context: clinical context typically:
        -- Provides background, rationale, or framing for the indications that follow
        -- Appears as plain text or paragraph(s), or as a title section before explicit guidelines
        -- May include definitions, epidemiological data, treatment overview, or references to clinical guidelines
        -- A clinical context belongs to all explicit medical guidelines that follow it
        
        Context Inheritance Rules:
        -- If a guideline is nested within another (i.e., a sub-guideline), its clinical context is defined by its parent guideline at the higher level of the hierarchy.
        -- If a set of medical guidelines are identified as sub-guidelines listed beneath another guideline, referred to as the parent medical guideline, then:
          --- Each sub-guideline should inherit the parent medical guideline itself as its clinical context.
          --- This means the parent guideline serves as the framing or rationale for all its nested sub-guidelines, regardless of depth.
        --  If there is no clinical context, just put `None`.
        -- Example:
        Advanced joint disease demonstrated by:
            • Radiographic supported evidence...
            •  Pain that cannot be adequately controlled...
            • If appropriate, history of unsuccessful conservative therapy... -- anti-inflammatory medications or analgesics, or
              - flexibility and muscle strengthening exercises, or
              - `Advanced joint disease demonstrated by:` is the clinical context for the three top-level guidelines.
            
            Example Explanation: The two nested guidelines under the third bullet point inherit their clinical context from their parent guideline, which is:
            `If appropriate, history of unsuccessful conservative therapy...`

              
  action:
  First please create a list of indications and limitations from {consolidated_mdgs} called consolidated mdg's in this prompt. The consolidated mdg's may contain multiple levels of nesting. The list should include all entries from every level, capturing the following fields: mdg_ID, mdg, and type. If a guideline is nested under another (e.g., bullet points under a parent bullet), treat it as part of a child group. Use hierarchical identifiers like mdg_1.1, mdg_1.1.1, etc., to reflect nesting depth.
  In the next step, given the mdg's in the created list, look at {LCD}, and according to the above-mentioned thought, organize each parent group of indications or limitations and their probable child groups using the dictionary structure below

    [{{
      'parent_group': {{
        'parent_id': '',
        'type': '',
        'logical_relation': '',
        'mdgs': [

          {{
            'member_id': '',
            'content': '',
            'clinical_context': ''
          }}
        ],

        'child_group': [
          {{
            'child_group_id': '',
            'logical_relation': '',
            'mdgs': [
              {{
                'member_id': '',
                'content': '',
                'clinical_context': ''
              }}
            ],
            child_group": [ ... ]  // Include this key to allow recursive nesting
          }}
        ]
      }}
    }}]

  please note that Each child group may itself contain further nested child groups. Ensure that nesting is preserved recursively.

  All parent groups should be returned as a list and each parent group should be a separate dictionary in the list, and the top-level key in each dictionary must be exactly 'parent_group'.  Do not rename the key to 'parent_group_2', 'parent_group_3', etc.—keep it constant.
  If there are multiple parent groups, return them as the Python list of dictionaries, each following the same structure. 
  Use **double quotes** for all keys and string values. Use `true`, `false`, `none` and `null` for boolean and none or null values.
  Return only a raw JSON array. Do not include any markdown formatting, triple backticks, or explanatory text. Output must be valid JSON only.  

"""


  # Define the prompts
  prompt_template_instruct = PromptTemplate(template=instruct_template, input_variables=[])
  prompt_template_contraindications_extraction = PromptTemplate(template=contraindications_extraction_prompt, input_variables=["LCD_title","LCD"])
  prompt_template_explicit_mdgs_extraction = PromptTemplate(template=explicit_mdgs_extraction_prompt, input_variables=["LCD"])
  prompt_template_implicit_mdgs_extraction = PromptTemplate(template=implicit_mdgs_extraction_prompt, input_variables=["LCD"])
  prompt_template_consolidating_mdgs = PromptTemplate(template=consolidating_mdgs_prompt, input_variables=["explicit_mdgs_extracted", "implicit_mdgs_extracted"])
  prompt_template_structure_extraction = PromptTemplate(template=structure_extraction_prompt, input_variables=["consolidated_mdgs"])

  # Define the chains using the pipe syntax
  chain_instruct = prompt_template_instruct | llm
  chain_contraindications_extraction = prompt_template_contraindications_extraction | llm
  chain_explicit_mdgs_extraction = prompt_template_explicit_mdgs_extraction | llm
  chain_implicit_mdgs_extraction = prompt_template_implicit_mdgs_extraction | llm
  chain_consolidated_mdgs = prompt_template_consolidating_mdgs | llm
  chain_structure_extraction = prompt_template_structure_extraction | llm


  # Ensure inputs are passed as dictionaries with the required keys
  result_instruct = chain_instruct.invoke({})
  result_contraindications = chain_contraindications_extraction.invoke({"LCD_title":LCD_title,"LCD":LCD})
  result_explicit = chain_explicit_mdgs_extraction.invoke({"LCD": LCD})
  result_implicit = chain_implicit_mdgs_extraction.invoke({"LCD": LCD})
  result_consolidated = chain_consolidated_mdgs.invoke({
      "explicit_mdgs_extracted": result_explicit.content,
      "implicit_mdgs_extracted": result_implicit.content
  })
  result_structure_extraction = chain_structure_extraction.invoke({"consolidated_mdgs":result_consolidated.content,"LCD":LCD})

  return extract_json_string(result_contraindications.content), extract_json_string(result_consolidated.content), extract_json_string(result_structure_extraction.content)


In [37]:
contraindications, LCD_mdgs, LCD_decision_tree = extract_LCD_medicalguidelines(LCD, LCD_title)

In [44]:
LCD_mdgs

[{'mdg_ID': 'mdg_1',
  'mdg': 'Radiographic supported evidence or when conventional radiography is not adequate, magnetic resonance imaging (MRI) and/or computed tomography (CT) (in situations when MRI is non-diagnostic or not able to be performed) supported evidence (subchondral cysts, subchondral sclerosis, periarticular osteophytes, joint subluxation, severe joint space narrowing, avascular necrosis); AND',
  'type': 'indication',
  'explicit': True,
  'justification': None,
  'extraction time': '2023-10-05 12:00:00'},
 {'mdg_ID': 'mdg_2',
  'mdg': 'Pain that cannot be adequately controlled despite optimal conservative treatment or functional disability from injury due to trauma or arthritis of the joint; AND',
  'type': 'indication',
  'explicit': True,
  'justification': None,
  'extraction time': '2023-10-05 12:00:00'},
 {'mdg_ID': 'mdg_3',
  'mdg': 'If appropriate, history of unsuccessful conservative therapy (non-surgical medical management) that is clearly addressed in the pre

In [45]:
LCD_decision_tree

[{'parent_group': {'parent_id': 'pg_1',
   'type': 'indication',
   'logical_relation': 'AND',
   'mdgs': [{'member_id': 'mdg_1',
     'content': 'Radiographic supported evidence or when conventional radiography is not adequate, magnetic resonance imaging (MRI) and/or computed tomography (CT) (in situations when MRI is non-diagnostic or not able to be performed) supported evidence (subchondral cysts, subchondral sclerosis, periarticular osteophytes, joint subluxation, severe joint space narrowing, avascular necrosis); AND',
     'clinical_context': 'Advanced joint disease demonstrated by:'},
    {'member_id': 'mdg_2',
     'content': 'Pain that cannot be adequately controlled despite optimal conservative treatment or functional disability from injury due to trauma or arthritis of the joint; AND',
     'clinical_context': 'Advanced joint disease demonstrated by:'},
    {'member_id': 'mdg_3',
     'content': 'If appropriate, history of unsuccessful conservative therapy (non-surgical medi

In [38]:
def extract_MCG_medicalguidelines(MCG,MCG_title):

  lead_in_sentence_logical_relation_prompt="""
    thought:
      MCG (Milliman Care Guidelines) documents include clinical indications in a hierarchical bullet-list format under a section titled “Clinical Indications.” The structure typically includes:
      - lead-in sentences: is an introductory statement that precedes the list of clinical indications in the MCG. They appears at the first level of a hierarchical tree structure and always mention the main procedure, entire or part of a MCG title, (e.g., "Lumbar MRI").
      - the lead-in sentence contains a logical term such `ALL`, or `one or more`.
    action:
       your task is to find the lead-in sentence in {MCG}, and extract the logical term. If it is `ALL', return `ALL`. If it is `pne or more`, return 'ANY'.
  
    OUTPUT:
      - only return the logical term. Never return lean-in sentence
      - Return a JSON object 
      - Use **double quotes** for all keys and strings.
      - Use `null` for null values.
      - Do NOT include markdown, code fences, or explanations—JSON only.
  """
              
  structure_extraction_prompt = """
    thought:
    MCG (Milliman Care Guidelines) documents include clinical indications in a hierarchical bullet-list format under a section titled “Clinical Indications.” The structure typically includes:
      - lead-in sentences: is an introductory statement that precedes the list of clinical indications in the MCG. They appears at the first level of a hierarchical tree structure and always mention the main procedure, entire or part of a MCG title, (e.g., "Lumbar MRI").
      -- A lead-in sentence is not a medical guidelines including indication, contraindication, or limitation.
      -- A lead-in sentence must not be considered directly or inderectly as a parent group or as a child group in the tree structure.
      -- Only take the logical relation mentioned in the lead-in sentence. This logical relation determine the logical relation between parent groups.
      
    - Top-level bullets representing major clinical scenarios (parent groups).
    - Nested bullets representing sub-criteria, often introduced by gate phrases like “as indicated by ALL of the following” or “as indicated by 1 or more of the following.”
    - Multiple levels of nesting (e.g., parent group → child group → sub-child group).
    - UI artifacts (Expand/Collapse, Supporting evidence links, reference markers) that must be removed.

    DEFINITIONS:
    - lead-in sentence: is an introductory statement that precedes the list of clinical indications in the MCG.
    - parent group: A first-level bullet under the lead-in sentence.
    - parent_id: Unique ID for each parent group (pg_1, pg_2, …).
    - child group: A nested set of indications under a parent or another child group, introduced by a gate phrase or age-specific header (e.g., “Adult and ALL of the following:”).
    - child_group_id: Unique per parent group (chg_1, chg_2, …).
    - type: Always "indication".
    - logical_relation: Derived from gate phrases:
        • “ALL of the following” → ALL
        • “1 or more of the following” / “one or more” → ANY
        • “NONE of the following” → NONE
        • “N or more of the following” → AT_LEAST_N (with n extracted)
    - mdgs: List of indications (criteria) under a group.
    - member_id: Hierarchical ID (e.g., mdg_3.2.1 for parent 3 → child 2 → item 1).
    - content: Cleaned text of the indication.
    - clinical_criteria: is a concise medical abbreviation or shortform that summarizes the clinical content. This should be 2-6 words maximum, use standard medical abbreviations when applicable, and focus on the key clinical concept. Examples:
        • "Diabetic macular edema" → "DME"
        • "No active intraocular inflammation" → "No Active Inflam"
        • "Radiographic supported evidence" → "Radiographic Evidence"  
        • "Pain or functional disability" → "Pain/Functional Disability"
        • "Clinical suspicion of device infection" → "Device Infection Suspicion"
        • "Asymptomatic patient with elevated troponin level" → "Elevated Troponin"
    - clinical_context: Introductory text that applies to the indication:
        • For top-level mdgs: the parent group text (minus gate phrase).
        • For nested mdgs: the immediate introducer text (minus gate phrase).
        • Inherit recursively for deeper nesting.

    TEXT CLEANING:
    - Preserve clinical meaning, parentheses, and examples (eg, ie).
    - Remove:
        • UI elements: “Expand”, “Collapse”, “Supporting evidence…”, “Return to top…”
        • Empty links  , reference markers ([n]), footnote letters ([A], [B], etc.)
    - Normalize whitespace; keep punctuation and numeric thresholds.

    CLINICAL CONTEXT RULES:
      - The clinical context of a node is the text of its immediate parent indication, which serves as the framing statement for that node and its siblings within the same logical block.
      - Each parent group becomes the clinical context for its child groups, and this relationship applies recursively for deeper nesting:

    TREE DATA STRUCTURE:
      - Follow this structure to generate the tree data structure:
        {{
        "parent_group": "pg_<index>",
        "type": "indication",
        "logical_relation": "ALL" | "ANY" | "NONE" | "AT_LEAST_N" | null,
        "mdgs": [
          {{
            "member_id": "mdg_<parentIndex>.<seq>[.<sub-seq>...]",
            "content": "<clean indication text>",
            "clinical_context": "<derived context per rules>",
            "clinical_criteria": "<2-6 word summary/abbr>"
          }}
        ],
        "child_group": [
          {{
            "child_group_id": "chg_<index>",
            "logical_relation": "ALL" | "ANY" | "NONE" | "AT_LEAST_N" | null,
            "mdgs": [ ... mdg items ... ],
            "child_group": [ ... recursive ... ]
          }}
        ]
      }}
      - Parent groups are defined as top-level clinical indications that appear immediately after the lead-in sentence.
      - If a Parent group doesn't have any children, "logical_relation": null and "mdgs"=[]
      - Child groups may contain nested children.
      - Lead-in sentences are excluded entirely—do not include them directly or indirectly in the tree.
    
      
  
    ID RULES:
    - parent_id: pg_1..pg_N in order of appearance.
    - member_id: mdg_<parentIndex>.<sequence>.<sub-sequence>...
    - child_group_id: chg_1.. per parent group.

    OUTPUT:
    - Return a JSON object 
    - Use **double quotes** for all keys and strings.
    - Use null for null values.
    - Do not include triple backticks or markdown formatting.

    ACTION:
    Given {MCG}, produce the full hierarchical structure as described above.

    """

  # Define the prompts
  prompt_template_lead_in_logic = PromptTemplate(template=lead_in_sentence_logical_relation_prompt,input_variables=["MCG"])
  prompt_template_structure_extraction = PromptTemplate(template=structure_extraction_prompt, input_variables=["MCG","MCG_title"])

  # Define the chains using the pipe syntax
  chain_lean_in_logic = prompt_template_lead_in_logic | llm
  chain_structure_extraction = prompt_template_structure_extraction | llm


  # Ensure inputs are passed as dictionaries with the required keys
  result_lead_in_logic = chain_lean_in_logic.invoke({"MCG":MCG})
  result_structure_extraction = chain_structure_extraction.invoke({"MCG":MCG,"MCG_title":MCG_title})

  
  try:
    root_logical_term = extract_json_string(result_lead_in_logic.content)[0]['logical_term']
  except (KeyError):
    root_logical_term = extract_json_string(result_lead_in_logic.content)['logical_term']
  
  
  MCG_tree_dict: Dict[str, str] = {
        "root": MCG_title,
        "logical_relation": root_logical_term,
        "parent_groups": extract_json_string(result_structure_extraction.content)
    }

  return  MCG_tree_dict


In [39]:
#MCG_decision_tree = extract_MCG_medicalguidelines(MCG,MCG_title)

In [16]:
MCG_title

NameError: name 'MCG_title' is not defined

In [17]:
MCG_decision_tree

NameError: name 'MCG_decision_tree' is not defined

## Question generation

In [40]:
question_generation_prompt = """
        Instruction:
        Please review the {medical_guideline}, which represents an indication or limitation extracted from a medical document such as a Local Coverage Determination (LCD) or Milliman Care Guidelines (MCG).
        Your task is to:
        - Generate at least one clear and concise question based on the type of medical guideline. The question should be designed to help determine whether relevant information related to the indication and its clinical context exists in a patient's medical record.
            -- Guideline types include: Lateral, Compound, Context-dependent, or Regular.
        - Enhance the generated question by incorporating all applicable clinical conditions listed in {clinical_context}. When doing so, ignore any structural or instructional phrases such as 'as indicated by', 'when all of the following are present', 'such as', or similar. Only include actual clinical conditions or medically relevant terms in the question.            -- If a clinical context entry does not contain any conditions, ignore it and do not include it in the question enhancement process.
        When generating the question:
            - Treat {medical_guideline} as the core subject.
            - Use {clinical_context} to narrow the domain of the question to relevant clinical conditions.
            - Ensure the final question reflects both the guideline and its associated clinical context.
        
        Thought:
        There are four main types of medical guidelines:

        1. Lateral Medical Guidelines  
        1.1. Definition: Involves an anatomical structure that can be lateralized. These include: eye, hip, knee, limb, shoulder, wrist, ankle, and others.
                        - If any of these anatomical parts are explicitly mentioned in the {medical_guideline} the guideline must be classified as 'lateral'.
                        - The follow-up clause must ask: 'If yes, left or right?' without repeating the anatomical part.
        1.2. Instruction:
                        Apply the principle of literality and include a follow-up clause that asks only for laterality — specifically, use the phrase: 'If yes, left or right?' Do not repeat the anatomical part in the follow-up clause. Only consider anatomical parts relevant to {medical_doc_title}.
        1.3. Example: 
                            Indication: Pain or functional disability from injury due to trauma or arthritis of the joint
                            Question: Does the patient experience pain in the knee or functional disability? If yes, which knee?
                        
                            Indication: Radiographic supported evidence or when conventional radiography is not adequate, magnetic resonance imaging (MRI) and/or computed tomography (CT) (in situations when MRI is non-diagnostic or not able to be performed) supported  evidence (subchondral cysts, subchondral sclerosis, periarticular osteophytes, joint subluxation, joint space narrowing, avascular necrosis)
                            Question:
                                    Has the patient undergone radiological exams such as X-ray, CT scan, or MRI on the knee? If yes, which knee?

        2. Compound Medical Guidelines  
        2.1. Definition: Includes multiple distinct clinical indications, each of which could independently justify medical necessity.  
        2.2. Instruction: perform the following steps:
                            Step 1: Identify and separate each distinct indication.
                            Step 2: Generate a medically relevant yes/no question for each distinct indication.

        2.3. Example:
                            Indication: Active urinary tract or dental infection
                            Question 1: Does the patient have active urinary tract infection?
                            Question 2: Does the patient have active dental infection?


        3. Context-Dependent Medical Guidelines  
        3.1. Definition: Includes a clinical indication that is only valid or relevant when a specific clinical context is met.  
        3.2. Instruction: perform the following:
                            - Identify the core indication.
                            - Extract any dependent clinical context.
                            - Incorporate complementary information already provided (e.g., duration of therapy).
                            - Generate two questions:
                            -- Question 1: one medically relevant yes/no question to the indication in order to see whether relevant information exists in a patient's medical record.
                            -- Question 2: one yes/no question relevant to the context to see if relevant information exists in a patient's medical record.
                        
        3.3. Example: 
                    Indication: supervised physical therapy [Activities of daily living (ADLs) diminished despite completing a plan of care,
                    yes/no Question: Has the patient undergone or completed physical therapy?
                    context Question: Has the patient undergone or completed a plan of care?

        4. Regular Medical Guidelines  
        4.1. Definition: Does not meet the criteria for lateral, compound, or context-dependent types.
        4.2. Instruction: generate only one yes/no question that can be answered by reviewing the medical records of a patient. The question should confirm whether relevant data or documentation exists for that indication.

        4.3. Examples:
                    Indication: Active urinary tract infection
                    Generated Question: Is there documentation of an active urinary tract infection in the medical record?
                    Indication: Use of anti-inflammatory medications
                    Generated Question: Is there evidence in the medical record that the patient is using anti-inflammatory medications?
                    Indication: Supervised physical therapy
                    Generated Question: Has the patient undergone supervised physical therapy as documented in the medical record?

        
        Action:
        Review `{medical_guideline}` and its clinical context, which are `{clinical_context}` and taken from `{medical_doc_title}`.  
        Follow these steps:
        First, determine the type of the medical guideline. Remember that a medical guideline might be lateral, compound, and context-dependent or a combination of these three types, but when it is regular, it cannot be the other three types.
        Second, determine the correct question generation logic based on the guideline type:
                - If the guideline is regular, it must not be treated as lateral, compound, or context-dependent. Only one question must be generated, even if multiple clinical conditions are present in {clinical_context}. These conditions should be combined into a single question.
                - If the guideline is compound, generate a separate question for each distinct indication.
                - If the guideline is context-dependent, generate one question for the indication and one for the context.
                - If the guideline is lateral, generate one question with a follow-up clause for laterality.
        Also, remember that the questions must:
            - determine whether the necessary information exists in the patient's medical record.
            - be specific to the medical guideline type identified in the previous step.
            - be clear, concise, and directly related to the medical guideline.
            - clearly address **all clinical conditions** listed in {clinical_context}.
        
        Lastly, remember if a medical guideline follows this pattern 'No <clinical condition>', the question should follow this pattern 'Is there an absence of <clinical condition> in the patient's record?'
        Example:
            - medical guideline: No neuromuscular disease (eg, myasthenia gravis)
            - Question: Is there an absence of neurological disease such as myasthemia gravis? 

        Each question must be returned as a dictionary with the following keys:

        - `"generated_question_id"`: A unique identifier for each question that follows the pattern of member_id. For example if 'member_id', 'mdg_2.1.1', then  'generated_question_id': 'Q_2.1.1'.
            -- If multiple questions are generated from a single guideline (e.g., two questions), then a numeric suffix is appended to the generated_question_id. For example, if the base ID is mdg_2.1.1, the questions will be labeled as Q_2.1.1.1 and Q_2.1.1.2
        - `"question"`: The generated question text.
        - `"question_type"`: One or a combination of `"lateral"`, `"compound"`, `"context-dependent"`, or only `"regular"`.
        - `"generation_time"`: The current timestamp in ISO 8601 format (e.g., `"2025-08-11T14:03:22Z"`).

        ---

        Output Format Instructions:

        - Return a **list of dictionaries**, each representing one question.
        - Use **valid JSON syntax** with double quotes.
        - Do **not** wrap the output in Markdown or code blocks.
        - Do **not** include any explanation, commentary, or extra text.
        """

In [41]:
#LLM-driven decision-tree augmentation:

"""
LLM-driven decision-tree augmentation:
- Finds leaf MDG nodes in your tree (prefix-based hierarchy).
- Builds clinical_context_chain from ancestor nodes.
- Calls LLM with your exact SystemMessage & HumanMessage templates.
- Inserts the returned list of question dicts after 'clinical_criteria' for each leaf.
- Returns the updated tree (dict).

Requires:
  - langchain-core (for SystemMessage, HumanMessage) or equivalent message classes.
  - An LLM instance exposing .invoke([SystemMessage(...), HumanMessage(...)]).

Author: You (revised with Copilot assistance)
"""

import ast
import json
import re
from collections import OrderedDict
from datetime import datetime, timezone
from typing import Dict, Any, List, Optional, Tuple


from langchain_core.messages import SystemMessage, HumanMessage



# ----------------------------
# Utilities: parsing & helpers
# ----------------------------

def parse_tree_from_text(text: str) -> Dict[str, Any]:
    """Safely parse the Python-literal tree file to a dict."""
    s = text.strip()
    try:
        return ast.literal_eval(s)
    except Exception:
        s2 = (s.replace("\u2264", "<=")
               .replace("\u2265", ">=")
               .replace("\u2013", "-")
               .replace("\u2014", "-"))
        return ast.literal_eval(s2)


def iter_groups(tree: Dict[str, Any]):
    """Yield all group dicts depth-first."""
    for pg in tree.get("parent_groups", []):
        yield pg
        for sub in _iter_child_groups(pg):
            yield sub

def _iter_child_groups(group: Dict[str, Any]):
    for sub in group.get("child_group", []):
        yield sub
        for sub2 in _iter_child_groups(sub):
            yield sub2

def collect_all_mdgs(tree: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    """Return {mdg_id: mdg_dict} for the entire tree."""
    by_id = {}
    for group in iter_groups(tree):
        for mdg in group.get("mdgs", []):
            by_id[mdg["member_id"]] = mdg
    return by_id

def find_leaf_mdg_ids(tree: Dict[str, Any]) -> Tuple[set, set]:
    """
    Detect leaf MDG IDs based on dot-prefix hierarchy in IDs.
    Returns (leaf_ids, non_leaf_ids).
    """
    all_ids = list(collect_all_mdgs(tree).keys())
    all_set = set(all_ids)
    non_leaf = set()

    for id1 in all_ids:
        prefix = id1 + "."
        if any(id2.startswith(prefix) for id2 in all_ids):
            non_leaf.add(id1)

    leaf = all_set - non_leaf
    return leaf, non_leaf

def mdg_ancestor_ids(mdg_id: str) -> List[str]:
    """
    For 'mdg_11.1.2.1' -> ['mdg_11', 'mdg_11.1', 'mdg_11.1.2'].
    Only returns prefixes shorter than the full id.
    """
    if not mdg_id.startswith("mdg_"):
        return []
    tail = mdg_id[len("mdg_"):]
    parts = tail.split(".")
    acc = []
    for i in range(1, len(parts)):
        acc.append("mdg_" + ".".join(parts[:i]))
    return acc

def build_clinical_context_chain(mdg_id: str,
                                 mdg_map: Dict[str, Dict[str, Any]]) -> str:
    """
    Build a readable clinical_context_chain string by looking at the leaf's own
    clinical_context and any meaningful ancestor info. We prioritize:
      - ancestor MDG 'content' (often expresses gating criteria),
      - ancestor MDG 'clinical_context' (if present),
      - leaf's own 'clinical_context'.
    Deduplicate and strip instructional parentheses like (eg ...).
    """
    def strip_examples(text: str) -> str:
        return re.sub(r"\((?:eg|i\.e\.|ie|such as)[^)]*\)",
                      "", text, flags=re.IGNORECASE).strip()

    contexts: List[str] = []

    # Ancestors first (more general), then leaf (more specific)
    for anc_id in mdg_ancestor_ids(mdg_id):
        anc = mdg_map.get(anc_id)
        if not anc:
            continue
        # prefer content as higher-signal context (e.g., "Arterial disease signs or symptoms")
        if anc.get("content"):
            contexts.append(strip_examples(anc["content"]))
        if anc.get("clinical_context"):
            contexts.append(strip_examples(anc["clinical_context"]))

    leaf = mdg_map[mdg_id]
    if leaf.get("clinical_context"):
        contexts.append(strip_examples(leaf["clinical_context"]))

    # Cleanup: normalize whitespace and de-dup while preserving order
    normalized = []
    seen = set()
    for c in contexts:
        c2 = re.sub(r"\s+", " ", c).strip().strip(",;")
        if c2 and c2.lower() not in seen:
            normalized.append(c2)
            seen.add(c2.lower())

    return "; ".join(normalized)



# -------------------------------------------------------
# Core: augment the tree using your LLM prompt & snippet
# -------------------------------------------------------

def augment_tree_with_llm(
    tree: Dict[str, Any],
    llm,  # must have .invoke([SystemMessage, HumanMessage])
    question_generation_prompt: str,
    medical_doc_title: Optional[str] = None,
    medical_doc_id: Optional[str] = None
) -> Dict[str, Any]:
    """
    Generates questions for each **leaf** MDG using the provided LLM and your prompt,
    and injects the resulting list under 'generated_questions' immediately
    after 'clinical_criteria'.

    Parameters
    ----------
    tree : dict
        Parsed decision tree (the structure in your 'decision tree structure.txt').
    llm : object
        Chat model exposing .invoke([...]) -> response with .content (LangChain or shim).
    question_generation_prompt : str
        The exact system prompt (from your 'question generation snippet.txt').
    medical_doc_title : str
        The document title; defaults to tree['root'] if omitted.
    medical_doc_id : str
        An identifier you use for the document (e.g., LCD ID). Optional.

    Returns
    -------
    dict
        Updated tree with 'generated_questions' inserted at each leaf MDG.
    """
    # Get book-keeping
    mdg_map = collect_all_mdgs(tree)
    leaf_ids, _ = find_leaf_mdg_ids(tree)
    system_message = SystemMessage(content=question_generation_prompt)

    # For reproducible timestamps across the run (LLM returns timestamp, but in case)
    iso_now = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")
    doc_title = medical_doc_title or tree.get("root", "")
    doc_id = medical_doc_id or ""

    # Build quick lookup from mdg_id to generated questions (list of dicts)
    questions_by_id: Dict[str, List[Dict[str, Any]]] = {}

    for mdg_id in sorted(leaf_ids):
        mdg = mdg_map[mdg_id]
        medical_guideline = mdg.get("content", "")
        clinical_context_chain = build_clinical_context_chain(mdg_id, mdg_map)

        # Compose user message (exactly as your snippet specifies)

        import textwrap

        user_prompt = textwrap.dedent(f"""
            Generate question for the following medical guideline and its clinical context:
            Medical Guideline: {medical_guideline}
            Clinical Context: {clinical_context_chain}
            document Title: {doc_title}
            document ID: {doc_id}
            Medical Guideline ID: {mdg_id}
        """).strip()


        human_message = HumanMessage(content=user_prompt)

        # Call LLM and parse JSON list
        response = llm.invoke([system_message, human_message])
        questions_generated = extract_json_string(response.content)

        # (Optional) augment missing generation_time if model didn't include it
        for i, q in enumerate(questions_generated, start=1):
            #q.setdefault("medical_guideline_id", mdg_id)
            q.setdefault("generated_question_id", f"Q_{i}")
            q.setdefault("generation_time", iso_now)

        questions_by_id[mdg_id] = questions_generated

    # Inject questions into MDG nodes, right after 'clinical_criteria'
    def inject_into_group(group: Dict[str, Any]):
        new_mdgs = []
        for mdg in group.get("mdgs", []):
            mid = mdg.get("member_id")
            if mid in questions_by_id:
                # Build a plain dict in the exact order required
                ordered_keys = ["member_id", "content", "clinical_context", "clinical_criteria"]
                new_mdg = {k: mdg[k] for k in ordered_keys if k in mdg}

                # Insert questions immediately after 'clinical_criteria'
                new_mdg["generated_questions"] = questions_by_id[mid]

                # Preserve any remaining keys in their original order
                for k, v in mdg.items():
                    if k not in new_mdg:
                        new_mdg[k] = v

                new_mdgs.append(new_mdg)
            else:
                # Not a target MDG—keep as-is
                new_mdgs.append(mdg)

        group["mdgs"] = new_mdgs

        # Recurse into children
        for sub in group.get("child_group", []):
            inject_into_group(sub)

    # Apply to all top-level parent groups
    for pg in tree.get("parent_groups", []):
        inject_into_group(pg)

    return tree



## Decisio Tree Generation

In [42]:
# decision tree generation
"""
Build an expandable/collapsible HTML view from an updated MCG decision tree.

Inputs
------
- A Python-literal dict file (single quotes are okay) named "updated tree structure.txt",
  or pass a custom path via --in.

What it renders
---------------
- Root title and logical relation.
- Parent groups as main branches; child groups as nested sub-branches.
- Each MDG node shows:
    - Medical Guideline (content)
    - Clinical Criteria (clinical_criteria)
- Leaf MDGs (those with 'generated_questions') also show the questions list.

Output
------
- Saves to: f"{MCG_title}_{MCG_ID}.html"
  - MCG_title comes from tree['root']
  - MCG_ID from tree['MCG_ID'] or tree['document_id']; if missing, uses --fallback-id (default "MCG_ID").

Notes
-----
- Uses <details><summary> for toggle UI; includes "Expand All" / "Collapse All" buttons.
"""

import ast
import html
import re
import argparse
from pathlib import Path
from typing import Any, Dict, List


# ---------- Parsing utilities ----------

'''def load_tree_dict(path: Path) -> Dict[str, Any]:
    """
    Load and parse the updated tree structure from a Python-literal file.
    Cleans away typical notebook "Output is truncated..." lines and stray ellipses.
    """
    raw = path.read_text(encoding="utf-8")

    # Remove notebook truncation boilerplate or stray ellipses lines
    lines = []
    for ln in raw.splitlines():
        if "Output is truncated" in ln or "Adjust cell output settings" in ln:
            continue
        if ln.strip() in {"...", "…"}:
            continue
        lines.append(ln)
    text = "\n".join(lines).strip()

    # Extract outermost {...} region
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]

    # Normalize some escaped characters commonly found in copied text
    text = (text
            .replace("\u2013", "-")
            .replace("\u2014", "-")
            .replace("\\n", "\n"))

    # Parse Python literal dict (handles single quotes)
    try:
        tree = ast.literal_eval(text)
    except Exception as e:
        raise ValueError(f"Failed to parse the tree file: {e}\n\nHead of text:\n{text[:600]}") from e

    if not isinstance(tree, dict):
        raise TypeError("Parsed content is not a dictionary at the root.")
    return tree

'''


def sanitize_filename(text: str, *, keep_hyphens: bool = True) -> str:
    """
    Sanitize a title/ID for filesystem use:
      - Remove parentheses and their contents: "(...)" -> ""
      - Replace slashes and similar separators with spaces
      - Collapse whitespace to single spaces
      - Remove disallowed chars (optionally keep hyphens)
      - Convert spaces to underscores
      - Collapse multiple underscores and trim
    """
    if text is None:
        return "MCG"

    s = str(text).strip()

    # Remove parentheses and their content, e.g., "(CTA)"
    s = re.sub(r"\([^)]*\)", "", s)

    # Replace common separators with spaces
    s = s.replace("/", " ").replace("\\", " ").replace("|", " ")

    # Normalize spaced hyphens (e.g., "AC - Abdominal") -> "AC - Abdominal"
    s = re.sub(r"\s*-\s*", " - ", s)  # unify spacing around hyphens for now

    # Collapse all whitespace to single spaces
    s = re.sub(r"\s+", " ", s).strip()

    # Allow alnum, space, underscore, and (optionally) hyphen
    if keep_hyphens:
        s = re.sub(r"[^A-Za-z0-9 _-]", "", s)
    else:
        s = re.sub(r"[^A-Za-z0-9 _]", "", s)

    # Convert spaced hyphens to either hyphen or underscore.
    # For titles, a cleaner look is often underscores; for IDs we keep hyphens.
    # We'll pass keep_hyphens=True for IDs and default for titles, so just keep as-is here.

    # Convert spaces to underscores
    s = s.replace(" ", "_")

    # Remove sequences like "_-_" or "-__-" if they occur
    s = re.sub(r"_*-_*(?=_|$)", "-", s)   # normalize stray combos
    s = re.sub(r"__+", "_", s)            # collapse multiple underscores

    # Trim leading/trailing underscores or hyphens
    s = s.strip("._-")

    return s or "MCG"



# ---------- HTML rendering helpers ----------

def esc(s: Any) -> str:
    return html.escape(str(s), quote=True) if s is not None else ""


def is_leaf_mdg(mdg: Dict[str, Any]) -> bool:
    """Leaf if it already contains generated questions."""
    return isinstance(mdg, dict) and isinstance(mdg.get("generated_questions"), list)


def render_mdg(mdg: Dict[str, Any]) -> str:
    member_id = esc(mdg.get("member_id", ""))
    content = esc(mdg.get("content", ""))
    clin_ctx = esc(mdg.get("clinical_context", ""))
    clin_crit = esc(mdg.get("clinical_criteria", ""))

    # Summary title: "mdg_id — content"
    title_parts = [f"<strong>{member_id}</strong>"]
    if content:
        title_parts.append(content)
    title = " — ".join(title_parts)

    inner: List[str] = []
    if clin_ctx:
        inner.append(f"<div class='kv'><span>Clinical Context:</span> <em>{clin_ctx}</em></div>")
    if clin_crit:
        inner.append(f"<div class='kv'><span>Clinical Criteria:</span> <code>{clin_crit}</code></div>")

    if is_leaf_mdg(mdg):
        qs = mdg.get("generated_questions", [])
        if qs:
            inner.append("<div class='kv'><span>Generated Questions:</span></div>")
            inner.append("<ul class='questions'>")
            for q in qs:
                qid = esc(q.get("generated_question_id", ""))
                qtext = esc(q.get("question", ""))
                qtype = esc(q.get("question_type", ""))
                qtime = esc(q.get("generation_time", ""))
                inner.append(
                    f"<li>"
                    f"<div><strong>{qid}</strong> — {qtext}</div>"
                    f"<div class='meta'>type: <code>{qtype}</code> • time: <code>{qtime}</code></div>"
                    f"</li>"
                )
            inner.append("</ul>")

    body = "\n".join(inner)
    return f"""
    <details class='mdg'>
      <summary>{title}</summary>
      <div class='mdg-body'>
        {body}
      </div>
    </details>
    """


def render_group(group: Dict[str, Any], level: int = 0) -> str:
    gid = esc(group.get("parent_id") or group.get("child_group_id") or "")
    gtype = esc(group.get("type", ""))
    logic = esc(group.get("logical_relation", ""))

    header_parts = []
    if gid:
        header_parts.append(f"<strong>{gid}</strong>")
    if gtype:
        header_parts.append(f"type: <code>{gtype}</code>")
    if logic:
        header_parts.append(f"logic: <code>{logic}</code>")
    header = " — ".join(header_parts) if header_parts else "Group"

    body_parts: List[str] = []
    for mdg in group.get("mdgs", []):
        body_parts.append(render_mdg(mdg))
    for sub in group.get("child_group", []):
        body_parts.append(render_group(sub, level + 1))

    body = "\n".join(body_parts)

    return f"""
    <details class='group level-{level}'>
      <summary>{header}</summary>
      <div class='group-body'>
        {body}
      </div>
    </details>
    """


def build_tree_html(tree: Dict[str, Any]) -> str:
    title = esc(tree.get("root", "MCG"))
    logic = esc(tree.get("logical_relation", ""))

    groups_html = [render_group(pg, 1) for pg in tree.get("parent_groups", [])]
    groups_joined = "".join(groups_html)

    return f"""<!doctype html>
<html lang='en'>
<head>
  <meta charset='utf-8'/>
  <meta name='viewport' content='width=device-width, initial-scale=1'/>
  <title>{title} — Tree</title>
  <style>
    :root {{
      --bg: #0b0f14; --panel: #121821; --ink: #e9eef6; --muted: #a6b3c6; --accent: #4aa3ff; --code: #ffd166;
      --border: #243043;
    }}
    html, body {{ background: var(--bg); color: var(--ink); font: 14px/1.5 system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif; }}
    body {{ margin: 0; padding: 0 0 4rem; }}
    header {{ position: sticky; top: 0; background: linear-gradient(180deg, rgba(11,15,20,.98), rgba(11,15,20,.92)); border-bottom: 1px solid var(--border); padding: 12px 16px; z-index: 2; }}
    h1 {{ margin: 0; font-size: 18px; }}
    .meta {{ color: var(--muted); }}
    main {{ padding: 16px; max-width: 1200px; margin: 0 auto; }}

    details {{ background: var(--panel); border: 1px solid var(--border); border-radius: 8px; margin: 8px 0; }}
    summary {{ cursor: pointer; padding: 10px 12px; outline: none; list-style: none; }}
    summary::-webkit-details-marker {{ display: none; }}
    summary::before {{ content: '▸'; display: inline-block; margin-right: 8px; color: var(--accent); transition: transform .15s ease; }}
    details[open] > summary::before {{ transform: rotate(90deg); }}

    .group-body, .mdg-body {{ padding: 0 12px 12px 12px; }}
    .kv > span {{ color: var(--muted); margin-right: 6px; }}
    code {{ color: var(--code); }}
    ul.questions {{ margin: 6px 0 0 20px; }}

    .controls {{ margin-top: 8px; display: flex; gap: 8px; }}
    button {{ background: var(--accent); color: #001433; border: 0; border-radius: 6px; padding: 6px 10px; font-weight: 600; cursor: pointer; }}
    button.secondary {{ background: transparent; color: var(--ink); border: 1px solid var(--border); }}
    .topline {{ margin-top: 4px; }}
  </style>
</head>
<body>
  <header>
    <h1>{title}</h1>
    <div class='meta topline'>Logical relation: <code>{logic}</code></div>
    <div class='controls'>
      <button onclick="toggleAll(true)">Expand All</button>
      <button class='secondary' onclick="toggleAll(false)">Collapse All</button>
    </div>
  </header>
  <main>
    {groups_joined}
  </main>
  <script>
    function toggleAll(open) {{
      document.querySelectorAll('details').forEach(d => d.open = open);
    }}
  </script>
</body>
</html>
"""


# ---------- CLI / Main ----------

def generate_tree_html(tree,MCG_ID):
    ap = argparse.ArgumentParser(description="Generate an HTML tree from an updated MCG tree structure.")
    ap.add_argument("--in", dest="in_path", default="updated tree structure.txt",
                    help="Input file containing the updated tree (default: 'updated tree structure.txt')")
    ap.add_argument("--outdir", dest="out_dir", default=".",
                    help="Directory to write the HTML file (default: current dir)")
    ap.add_argument("--fallback-id", dest="fallback_id", default="MCG_ID",
                    help="Fallback MCG_ID when not present in the tree (default: 'MCG_ID')")
    args = ap.parse_args()

    #in_path = Path(args.in_path).expanduser().resolve()
    out_dir = Path(args.out_dir).expanduser().resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    #tree = load_tree_dict(in_path)

    
    # Derive the pieces (assumes you already parsed `tree` dict)
    mcg_title_raw = tree.get("root", "MCG")
    

    # Sanitize:
    # - For titles, prefer removing hyphens introduced only as spacers (convert to underscores).
    # - For IDs, keep hyphens (e.g., AC-CTA-001).
    mcg_title = sanitize_filename(mcg_title_raw, keep_hyphens=False)
    mcg_id = sanitize_filename(MCG_ID, keep_hyphens=True)


    html_doc = build_tree_html(tree)

    
    # Assume mcg_title, mcg_id, html_doc are already defined
    # If you have an out_dir variable, use it; otherwise, default to current directory
    out_dir = Path(".")  # or replace with your desired base directory

    # Create the 'tree' subfolder inside out_dir
    tree_dir = out_dir / "trees"
    tree_dir.mkdir(parents=True, exist_ok=True)

    # Build the output file name and path
    out_name = f"{mcg_title}_{mcg_id}.html"
    out_path = tree_dir / out_name

    # Write the HTML content to the file
    out_path.write_text(html_doc, encoding="utf-8")
    
    
    # Ensure 'jsons' folder exists
    json_dir = Path("jsons")
    json_dir.mkdir(parents=True, exist_ok=True)

    # Build output path
    out_name = f"{mcg_title}_{mcg_id}.json"
    out_path = json_dir / out_name

    # Dump JSON
    with open(out_path, "w", encoding="utf-8") as f:
      json.dump(tree, f, indent=2, ensure_ascii=False)


    print(f"Wrote: {out_path}")

In [21]:
import glob

html_files = glob.glob("*.html")

from tqdm import tqdm

for filename in tqdm(html_files[20:]):
    MCG, MCG_title, MCG_ID = fetch_MCG(filename)
    MCG_decision_tree = extract_MCG_medicalguidelines(MCG,MCG_title)
    MCG_decision_tree_updated = augment_tree_with_llm(
        tree=MCG_decision_tree,
        llm=llm,
        question_generation_prompt=question_generation_prompt,
        medical_doc_title= MCG_title,
        medical_doc_id=MCG_ID
        )
    generate_tree_html(MCG_decision_tree_updated,MCG_ID)

0it [00:00, ?it/s]


## Questions Evaluation

In [43]:
def questions_evaluation(questions_df):
    
    evaluation_df = pd.DataFrame(columns = ['question_id', 'relevancy', 'relevancy_justification', 'accuracy', 'accuracy_justification', 'clarity', 'clarity_justification', 'completeness', 'completeness_justification'])

    question_evaluation_prompt = ''' Please evaluate this question, {question}, to see how much it is aligned to {medical_guideline} and all {clinical_context}, based on the below criteria:
    1. Relevancy
    Definition: Measures how directly the question relates to the specific medical guideline being referenced. A highly relevant question will reflect the intent, scope, and clinical context of the guideline.
    Why it matters: Ensures the question is grounded in the correct clinical framework and supports guideline-based decision-making.
    2. Accuracy
    Definition: Assesses whether the question uses correct medical terminology, reflects current clinical standards, and avoids factual errors. It should be consistent with established medical knowledge and policy concepts.
    Why it matters: Prevents misinterpretation and ensures safe, evidence-based care.
    3. Clarity
    Definition: Evaluates whether the question is clearly worded and interpretable in only one way. It should avoid vague language, double meanings, or overly complex phrasing.
    Why it matters: Reduces confusion and ensures consistent understanding across reviewers or systems.
    4. Completeness
    Definition: Determines whether the question includes all necessary qualifiers, conditions, and context to make a guideline-based decision. It should not omit critical information that could affect the outcome.
    Why it matters: Supports comprehensive evaluation and minimizes the risk of incorrect or incomplete decisions.

    Please use this rubric to score the criteria:
    0: 	Completely unrelated to the guideline; no identifiable connection.
    0.25: 	Minimally related; vague or tangential reference to the guideline.
    0.5: 	Partially related; some relevant elements but lacks direct alignment.
    0.75:	Mostly related; aligns with the guideline but misses some nuance or scope.
    1:	Fully aligned; clearly reflects the intent, scope, and clinical context of the guideline.

    In the end, please return the result in the form of this sictionary:
    {{'question_id': {question_id},
      'question': question,
      'relevancy': relevancy_score,
      'relevancy_justification': explain your justification about relevancy score here,
      'accuracy': accuracy_score,
      'accuracy_justification': explain your justification about accuracy score here,
      'clarity':clarity_score,
      'clarity_justification': explain your justification about clarity score here,
      'completeness': Completeness_score
      'completeness_justification': explain your justification about completeness score here,
    }}

   
    output format:
    - Return only as **valid JSON syntax** with double quotes 
    - Do **not** wrap the output in Markdown or code blocks.
    - Do **not** include any explanation, commentary, or extra text.
    '''
    
    system_message = SystemMessage(content= question_evaluation_prompt)

    for _, row in questions_df.iterrows():
        question_id = row['question_id']
        question = row['Question']
        medical_guideline = row['Medical Guideline']
        clinical_context = row['Clinical Context']
        
       
        user_message = HumanMessage(content=f"""
                        Evaluate the following question based on the rubric:

                        question_id: {question_id}
                        Question: {question}
                        Medical Guideline: {medical_guideline}
                        Clinical Context: {clinical_context}
                        """)


        # Call the LLM
        response = llm.invoke([system_message, user_message])
        scores = extract_json_string(response.content)

        del scores['question']
        
        new_df = pd.DataFrame([scores])
        # Only concatenate if `scores` is not empty or all-NA
        if not new_df.isna().all(axis=1).all():
            evaluation_df = pd.concat([evaluation_df, new_df], ignore_index=True)

    evaluation_df['confidence_score'] = (evaluation_df.relevancy+evaluation_df.accuracy+evaluation_df.clarity+evaluation_df.completeness)/4

    evaluation_df['justifications_summary'] = [get_completion(f'summarize {row.relevancy_justification} and {row.accuracy_justification} and {row.clarity_justification} and {row.completeness_justification}')
        for _,row in evaluation_df.iterrows()
        ]
    
    evaluation_df['confidence_level'] = ['High' if score >= 0.75 else
                                         'Medium' if score >= 0.5 else
                                         'Low'
                                        for score in evaluation_df['confidence_score']
                                        ]


    return evaluation_df

In [23]:
MCG_evaluation_df = questions_evaluation(MCG_questions_df)

NameError: name 'MCG_questions_df' is not defined

In [24]:
MCG_evaluation_df

NameError: name 'MCG_evaluation_df' is not defined

In [25]:
questions_evaluated_df = pd.merge(MCG_questions_df,MCG_evaluation_df, on='question_id', how='left')

NameError: name 'MCG_questions_df' is not defined

In [26]:
questions_evaluated_df.to_csv('AC - Abdominal_evaluation.csv',index=False)

NameError: name 'questions_evaluated_df' is not defined

In [27]:
questions_evaluated_df.to_csv('test.csv', index=False)

NameError: name 'questions_evaluated_df' is not defined